# 01 Governance

Own the complete Governance lifecycle in one persistent notebook: establish steward and agreement context before engineering, then return after `02_pipeline` produces catalogue and profile evidence to register contracts, read that evidence, enrich metadata, and define guardrails for the ETL workflow.

Required delivery flow: **Governance → Engineering → Governance** (`01_governance` → `02_pipeline` → `01_governance`). Optional support remains in `99_explore`.

FabricOps uses standalone widget cells because smaller widget outputs are more stable in Microsoft Fabric notebooks.

## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release  | Tested by | Date tested | 
|---|---|---| 
|-|  - | -  | 


## 1. Run `00_env_config`

In [ ]:
%run 00_env_config

## 2. Import required Governance functions

In [ ]:
from fabricops_kit import (
    # FabricOps v0.1.0 onwards
    widget_author_dq_rules,
    widget_author_guardrails,
    widget_enrich_table_metadata,
    widget_register_data_contract,
    widget_render_data_agreement,
    widget_render_data_steward,
    widget_view_agreement_catalogue,
)

## 3. Data Steward

Run this standalone cell to create or update Data Steward metadata. Run it before establishing a Data Agreement so the accountable parties are available for selection.

In [ ]:
steward_widget = widget_render_data_steward(spark=spark)

## 4. Data Agreement

Create or select the overarching governance agreement between accountable producer and consumer stewards. The agreement defines purpose, scope, ownership, permitted use, and governance conditions. Run this standalone cell after at least one active steward exists.

In [ ]:
agreement_widget = widget_render_data_agreement(spark=spark)

## 5. Data Contract

After `02_pipeline` has produced catalogue and validation evidence, return to this notebook and register one or more machine-readable Data Contracts under the selected Data Agreement. Contract membership is stored at the logical dataset level and metadata routing is resolved from `00_env_config`.

In [ ]:
contract_state = widget_register_data_contract(
    agreement=agreement_widget,
    target="metadata",
    spark_session=spark,
)

## 6. Catalogue and profiling evidence

Select a dataset linked to the current agreement through its registered Data Contracts. The active environment determines which catalogue and profile observations are reviewed.

In [ ]:
agreement_catalogue_view = widget_view_agreement_catalogue(
    agreement=agreement_widget,
    target="metadata",
    spark_session=spark,
)

In [ ]:
views = agreement_catalogue_view["get_views"]()
catalogue_df = views["catalogue"]
profile_df = views["profile"]
frequency_df = views["frequency"]

In [ ]:
display(catalogue_df)

In [ ]:
display(profile_df)
display(frequency_df)

## 7. Select governed dataset

Select a profiled source table or pipeline output from the metadata catalogue. Run `02_pipeline` profiling first if the dataset is not available.

## 8. Enrichment

Review and update table and column context, classifications, and other enrichment details for the selected dataset.

In [ ]:
enrichment_state = widget_enrich_table_metadata(
    spark_session=spark,
)

## 9. Guardrail authoring

Create or update schema, freshness, profile-behaviour, and data-quality guardrail rules. Governance defines the guardrails that Engineering re-validates in `02_pipeline`.

In [ ]:
guardrail_authoring_state = widget_author_guardrails(
    spark_session=spark,
)

In [ ]:
dq_authoring_state = widget_author_dq_rules(
    spark_session=spark,
    source_notebook_type="01_governance",
    created_by_role="governance",
)